<a href="https://colab.research.google.com/github/PushpDayalMathur/PushpDayalMathur/blob/SplCharCheck/CA_CPRP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
import os

# Remove uploaded files to ensure fresh upload on next run
if 'stage1_filename' in globals() and os.path.exists(stage1_filename):
    os.remove(stage1_filename)
    print(f"Removed {stage1_filename}")

if 'metadata_filename' in globals() and os.path.exists(metadata_filename):
    os.remove(metadata_filename)
    print(f"Removed {metadata_filename}")


# Upload files
uploaded_stage1 = files.upload()
stage1_filename = next(iter(uploaded_stage1.keys()))

uploaded_metadata = files.upload()
metadata_filename = next(iter(uploaded_metadata.keys()))

# Read ModelManager sheet and select columns
df_model_manager = pd.read_excel(stage1_filename, sheet_name='ModelManager')
df_curve_parameters = df_model_manager[["VariableName", "Type", "Transformation", "LagMin","Decay", "DecayMin"]]

# Exclude rows with 'Linear' transformation
df_curve_parameters = df_curve_parameters[df_curve_parameters['Transformation'] != 'Linear']

# Read Activity Spends Mapping sheet and select columns
df_activity_spends_mapping = pd.read_excel(metadata_filename, sheet_name='Activity Spends Mapping')
df_spends_metric = df_activity_spends_mapping[['Activity', 'Spends']]

# Filter spends for variables in df_curve_parameters
spends_for_curve_variables = df_spends_metric[df_spends_metric['Activity'].isin(df_curve_parameters['VariableName'])]

# Merge dataframes
merged_df = pd.merge(df_curve_parameters, spends_for_curve_variables, left_on='VariableName', right_on='Activity', how='left')

# Drop the 'Activity' column as it's redundant after the merge
merged_df = merged_df.drop(columns=['Activity'])

# Fill NaN values in 'Spends' with 'VariableName'
merged_df['Spends'] = merged_df['Spends'].fillna(merged_df['VariableName'])

# Read the 'AFRawData' sheet from the stage 1 dump file
df_raw_data = pd.read_excel(stage1_filename, sheet_name='AFRawData')

# Read the 'Contribution' sheet from the stage 1 dump file
df_contribution = pd.read_excel(stage1_filename, sheet_name='Contribution')


# Determine periodicity based on the 'Period' column in df_contribution
if 'Period' in df_contribution.columns:
    df_contribution['Period'] = pd.to_datetime(df_contribution['Period'])
    period_diff = df_contribution['Period'].diff().dropna().mode()

    if not period_diff.empty:
        if period_diff[0] == pd.Timedelta(days=1):
            periodicity = 'Daily'
        elif period_diff[0] == pd.Timedelta(days=7):
            periodicity = 'Weekly'
        else:
            periodicity = 'Monthly' # Assuming anything more than 7 days is monthly
    else:
        periodicity = 'Unknown' # Handle case with only one period

else:
    periodicity = 'Unknown (Period column missing)' # Handle case where Period column is missing

# Add 'Periodicity' column to merged_df
merged_df['Periodicity'] = periodicity

# Display the final merged dataframe
#display(merged_df)

# Get the list of Variable Names and Spends
variable_names_list = merged_df['VariableName'].tolist()
spends_list = merged_df['Spends'].tolist()

# Get the list of periods from the contribution data
periods_to_keep = df_contribution['Period'].unique().tolist()

# Function to create dataframes based on a list of columns
def create_filtered_dataframe(source_df, column_list, period_list):
    filtered_df = source_df[source_df['Period'].isin(period_list)][['Period']].copy()
    for col in column_list:
        if col in source_df.columns:
            filtered_df[col] = source_df[source_df['Period'].isin(period_list)][col]
        else:
            filtered_df[col] = None
    return filtered_df

# Create the dataframe for VariableName data
variable_name_data = create_filtered_dataframe(df_raw_data, variable_names_list, periods_to_keep)

# Create the dataframe for Spends data
spends_data = create_filtered_dataframe(df_raw_data, spends_list, periods_to_keep)


# Display the first few rows of the new dataframes
display("VariableName Data:")
#display(variable_name_data.head())

display("\nSpends Data:")
#display(spends_data.head())

# Get the list of Variable Names from the merged_df
variable_names_list = merged_df['VariableName'].tolist()

# Create a list of columns to keep from the Contribution sheet, including 'Period'
# and only the columns from variable_names_list that exist in df_contribution
contribution_columns_to_keep = ['Period'] + [col for col in variable_names_list if col in df_contribution.columns]

# Create the contribution data dataframe with selected columns
contribution_data_filtered = df_contribution[contribution_columns_to_keep]

# Display the first few rows of the new dataframe
display("Contribution Data:")
#display(df_contribution.head())

# Calculate the date range for the "Full Period"
full_period_start = contribution_data_filtered['Period'].min()
full_period_end = contribution_data_filtered['Period'].max()

print(f"Full Period: {full_period_start} to {full_period_end}")

# Determine the start date for the "Last 1 Year" based on periodicity
last_year_end = full_period_end

if periodicity == 'Weekly':
    last_year_start = last_year_end - pd.Timedelta(weeks=52)
elif periodicity == 'Monthly':
    last_year_start = last_year_end - pd.DateOffset(months=12)
else:
    last_year_start = None # Handle unknown periodicity case

print(f"Last 1 Year Period: {last_year_start} to {last_year_end}")

# Create a new DataFrame called variable_name_data_full_period
variable_name_data_full_period = variable_name_data[
    (variable_name_data['Period'] >= full_period_start) &
    (variable_name_data['Period'] <= full_period_end)
].copy()

# Create a new DataFrame called variable_name_data_last_year
variable_name_data_last_year = variable_name_data[
    (variable_name_data['Period'] >= last_year_start) &
    (variable_name_data['Period'] <= last_year_end)
].copy()

# Create a new DataFrame called spends_data_full_period
spends_data_full_period = spends_data[
    (spends_data['Period'] >= full_period_start) &
    (spends_data['Period'] <= full_period_end)
].copy()

# Create a new DataFrame called spends_data_last_year
spends_data_last_year = spends_data[
    (spends_data['Period'] >= last_year_start) &
    (spends_data['Period'] <= last_year_end)
].copy()

# Display the first few rows of the new dataframes to verify
display("VariableName Data - Full Period:")
#display(variable_name_data_full_period.head())

display("\nVariableName Data - Last Year:")
#display(variable_name_data_last_year.head())

display("\nSpends Data - Full Period:")
#display(spends_data_full_period.head())

display("\nSpends Data - Last Year:")
#display(spends_data_last_year.head())


def calculate_aol(df):
    """Calculates the average of non-zero values for each column in a DataFrame."""
    # Exclude the 'Period' column
    df_numeric = df.drop(columns=['Period'], errors='ignore')

    # Replace zeros with NaN to exclude them from the mean calculation
    df_numeric = df_numeric.replace(0, pd.NA)

    # Calculate the mean, skipping NaN values
    aol_values = df_numeric.mean(skipna=True)

    return aol_values

def calculate_sum(df):
    """Calculates the sum for each column in a DataFrame."""
    # Exclude the 'Period' column
    df_numeric = df.drop(columns=['Period'], errors='ignore')

    # Calculate the sum
    sum_values = df_numeric.sum()

    return sum_values

# Calculate AOL for the full period and last year
aol_full_period = calculate_aol(variable_name_data_full_period)
aol_last_year = calculate_aol(variable_name_data_last_year)

# Calculate sum of activity for the full period and last year
activity_sum_full_period = calculate_sum(variable_name_data_full_period)
activity_sum_last_year = calculate_sum(variable_name_data_last_year)

# Calculate sum of spends for the full period and last year
spends_sum_full_period = calculate_sum(spends_data_full_period)
spends_sum_last_year = calculate_sum(spends_data_last_year)

# Display the calculated metrics
display("Average Operating Level (AOL) - Full Period:")
#display(aol_full_period)

display("\nAverage Operating Level (AOL) - Last Year:")
#display(aol_last_year)

display("\nActivity Sum - Full Period:")
#display(activity_sum_full_period)

display("\nActivity Sum - Last Year:")
#display(activity_sum_last_year)

display("\nSpends Sum - Full Period:")
#display(spends_sum_full_period)

display("\nSpends Sum - Last Year:")
#display(spends_sum_last_year)

# Calculate CPRP for the full period, handling division by zero
# Ensure both series have the same index for correct division
cprp_full_period = spends_sum_full_period.divide(activity_sum_full_period).replace([float('inf'), -float('inf')], 0)

# Calculate CPRP for the last year, handling division by zero
# Ensure both series have the same index for correct division
cprp_last_year = spends_sum_last_year.divide(activity_sum_last_year).replace([float('inf'), -float('inf')], 0)


# Display the calculated CPRPs with increased precision
print("CPRP - Full Period:")
#display(cprp_full_period.apply(lambda x: f'{x:.16f}' if pd.notna(x) else 'NaN'))

print("\nCPRP - Last Year:")
#display(cprp_last_year.apply(lambda x: f'{x:.16f}' if pd.notna(x) else 'NaN'))

# Create new columns in merged_df by mapping the calculated metrics
merged_df['AOL_Full_Period'] = merged_df['VariableName'].map(aol_full_period)
merged_df['AOL_Last_Year'] = merged_df['VariableName'].map(aol_last_year)
merged_df['Activity_Sum_Full_Period'] = merged_df['VariableName'].map(activity_sum_full_period)
merged_df['Activity_Sum_Last_Year'] = merged_df['VariableName'].map(activity_sum_last_year)
merged_df['Spends_Sum_Full_Period'] = merged_df['Spends'].map(spends_sum_full_period)
merged_df['Spends_Sum_Last_Year'] = merged_df['Spends'].map(spends_sum_last_year)

# Calculate CPRP directly in merged_df after adding the sum columns
# Handle division by zero by filling resulting infinity/NaN values with 0 or another appropriate value
merged_df['CPRP_Full_Period'] = merged_df['Spends_Sum_Full_Period'].divide(merged_df['Activity_Sum_Full_Period']).replace([float('inf'), -float('inf'), pd.NA], 0)
merged_df['CPRP_Last_Year'] = merged_df['Spends_Sum_Last_Year'].divide(merged_df['Activity_Sum_Last_Year']).replace([float('inf'), -float('inf'), pd.NA], 0)


# Display the updated merged_df
#display(merged_df)


# Define the output filename
output_filename = 'Final_Output.xlsx'

# Create an ExcelWriter object
with pd.ExcelWriter(output_filename) as writer:
    # Write each dataframe to a different sheet
    merged_df.to_excel(writer, sheet_name='Merged Data', index=False)
    variable_name_data.to_excel(writer, sheet_name='VariableName Data', index=False)
    spends_data.to_excel(writer, sheet_name='Spends Data', index=False)
    contribution_data_filtered.to_excel(writer, sheet_name='Contribution Data', index=False)


print(f"Dataframes exported to {output_filename}")

# Download the file
files.download('Final_Output.xlsx')

# Remove uploaded files to ensure fresh upload on next run
if 'stage1_filename' in globals() and os.path.exists(stage1_filename):
    os.remove(stage1_filename)
    print(f"Removed {stage1_filename}")

if 'metadata_filename' in globals() and os.path.exists(metadata_filename):
    os.remove(metadata_filename)
    print(f"Removed {metadata_filename}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')